In [9]:
import json 
import os 
import math

import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt
import tiktoken

from datasets import Dataset

from importlib.metadata import version

In [10]:

# from datasets import load_dataset

# dataset = load_dataset("tatsu-lab/alpaca")

# print(dataset)


In [11]:
dataset = Dataset.from_file(
    "../data/instruction_tuning/alpaca-train.arrow"
)

print(f"Original dataset rows: {len(dataset):,}")
print(dataset)


Original dataset rows: 52,002
Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 52002
})


In [12]:
tokenizer = tiktoken.get_encoding("gpt2")


In [13]:
TARGET_TOKENS = 3_000_000
running_tokens = 0
cutoff_index = 0

for i, row in enumerate(dataset):
    num_tokens = len(tokenizer.encode(row["text"], allowed_special={"<|endoftext|>"}))
    running_tokens += num_tokens
    if running_tokens >= TARGET_TOKENS:
        cutoff_index = i + 1
        break

dataset = dataset.select(range(cutoff_index))

total_character = sum(len(entry["text"]) for entry in dataset)
print(f"Truncated rows   : {len(dataset):,}")
print(f"Total characters : {total_character:,}")
print(f"Total tokens     : {running_tokens:,}")


Truncated rows   : 27,054
Total characters : 13,988,748
Total tokens     : 3,000,023


In [14]:
print("First sample instruction and response:\n")
print(dataset[0]["text"])


First sample instruction and response:

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.
